# Chess HalfKP NNUE - Colab trainer

**Before running:** Runtime -> Change runtime type -> Hardware accelerator = **GPU (T4)**.
Then Runtime -> **Run all**.

This downloads one shard (~31.6M positions = your 10% slice), preprocesses it,
trains the HalfKP net on the GPU, exports `net.nnue` in the engine's format, and
downloads it to your computer. Architecture + feature indexing match the repo's
`training/` files exactly, so the file drops straight into the engine.

Tweak the knobs in the **Config** cell (LIMIT / EPOCHS) to trade time vs quality.


In [ ]:
# 1. Check the GPU (must say CUDA: True). If not, set Runtime -> GPU and re-run.
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU - set Runtime to GPU!)")

In [ ]:
# 2. Deps (torch/numpy already on Colab)
!pip -q install huggingface_hub pyarrow

In [ ]:
# 3. Config + HalfKP feature indexing (IDENTICAL to engine/nn/nnue.cpp & training/halfkp.py)
SHARD  = "train-00000.parquet"   # one shard = ~31.6M positions
LIMIT  = 12_000_000              # positions to use (0 = all ~31.6M). Lower = faster.
EPOCHS = 6
BATCH  = 16384

NUM_FEATURES = 64 * 10 * 64      # 40960
ACC, L1, L2  = 256, 32, 32
EVAL_SCALE   = 400.0
CLAMP        = 2000              # clamp |cp|; mate -> +-CLAMP

PIECE = {'P':(0,0),'N':(0,1),'B':(0,2),'R':(0,3),'Q':(0,4),'K':(0,5),
         'p':(1,0),'n':(1,1),'b':(1,2),'r':(1,3),'q':(1,4),'k':(1,5)}

def parse_fen(fen):
    parts = fen.split(); board, stm = parts[0], parts[1]
    pieces = []
    for r, row in enumerate(board.split('/')):
        rank = 7 - r; f = 0
        for ch in row:
            if ch.isdigit(): f += int(ch)
            else:
                c, pt = PIECE[ch]; pieces.append((c, pt, rank*8+f)); f += 1
    return pieces, (0 if stm == 'w' else 1)

def orient(persp, s): return s if persp == 0 else (s ^ 56)

def features(pieces, persp):
    king = next(sq for (c,pt,sq) in pieces if pt == 5 and c == persp)
    ok = orient(persp, king); out = []
    for (c, pt, sq) in pieces:
        if pt == 5: continue
        rel = 0 if c == persp else 1
        out.append(ok*(10*64) + (rel*5+pt)*64 + orient(persp, sq))
    return out
print("config + halfkp ready; LIMIT =", LIMIT)

In [ ]:
# 4. Download ONE shard (~726MB) to /content (NOT load_dataset - that grabs all 10)
import os
os.environ['HF_HOME'] = '/content/.hf'
from huggingface_hub import hf_hub_download
pq = hf_hub_download('mateuszgrzyb/lichess-stockfish-normalized', SHARD,
                     repo_type='dataset', local_dir='/content/data')
print("downloaded:", pq, round(os.path.getsize(pq)/1e6), "MB")

In [ ]:
# 5. Preprocess -> packed binary (memmapped for fast training). One-time.
import numpy as np, struct, math, time
import pyarrow.parquet as pq_reader

MAXF = 30
REC  = 4 + 2 + MAXF*2*2           # 126 bytes: f32 target | u8 scnt,ocnt | u16 stm[30],opp[30]

def make_record(fen, cp_white):
    try: pieces, stm = parse_fen(fen)
    except Exception: return None
    wk = sum(1 for c,pt,_ in pieces if pt==5 and c==0)
    bk = sum(1 for c,pt,_ in pieces if pt==5 and c==1)
    if wk!=1 or bk!=1 or len(pieces)>32: return None
    fs, fo = features(pieces, stm), features(pieces, stm^1)
    if len(fs)>MAXF or len(fo)>MAXF: return None
    cpw = max(-CLAMP, min(CLAMP, cp_white))
    wp = 1.0/(1.0+math.exp(-cpw/EVAL_SCALE))
    target = wp if stm==0 else (1.0-wp)
    s = np.zeros(MAXF, np.uint16); s[:len(fs)] = fs
    o = np.zeros(MAXF, np.uint16); o[:len(fo)] = fo
    return struct.pack('<f', target) + struct.pack('<BB', len(fs), len(fo)) + s.tobytes() + o.tobytes()

out_path = '/content/train.bin'
n = 0; t0 = time.time()
pf = pq_reader.ParquetFile(pq)
with open(out_path, 'wb') as out:
    done = False
    for batch in pf.iter_batches(batch_size=65536, columns=['fen','cp','mate']):
        d = batch.to_pydict(); fens, cps, mates = d['fen'], d['cp'], d['mate']
        for i in range(len(fens)):
            cp, mate = cps[i], mates[i]
            if cp is not None: cpw = float(cp)
            elif mate is not None: cpw = CLAMP if mate > 0 else -CLAMP
            else: continue
            rec = make_record(fens[i], cpw)
            if rec is not None:
                out.write(rec); n += 1
                if n % 2_000_000 == 0:
                    print(f"  {n:,} positions  ({n/(time.time()-t0):,.0f}/s)")
                if LIMIT and n >= LIMIT: done = True; break
        if done: break
open(out_path+'.meta','w').write(f"{n} {MAXF} {REC}\n")
print(f"packed {n:,} positions -> {out_path}  ({n*REC/1e9:.2f} GB)  in {time.time()-t0:.0f}s")

In [ ]:
# 6. Model (EmbeddingBag = the accumulator) + memmapped dataset.  Matches the engine.
import torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader

def crelu(x): return torch.clamp(x, 0.0, 1.0)

class NNUE(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.EmbeddingBag(NUM_FEATURES, ACC, mode='sum')
        self.b1 = nn.Parameter(torch.zeros(ACC))
        self.l1 = nn.Linear(2*ACC, L1); self.l2 = nn.Linear(L1, L2); self.l3 = nn.Linear(L2, 1)
        nn.init.normal_(self.transformer.weight, std=0.01)
    def forward(self, si, so, oi, oo):
        a_s = crelu(self.transformer(si, so) + self.b1)
        a_o = crelu(self.transformer(oi, oo) + self.b1)
        x = torch.cat([a_s, a_o], dim=1)
        return self.l3(crelu(self.l2(crelu(self.l1(x))))).squeeze(1)
    def win_prob(self, out): return torch.sigmoid(out / EVAL_SCALE)

def read_meta(p):
    n, maxf, rec = map(int, open(p+'.meta').read().split()); return n, maxf, rec

class PackedDataset(Dataset):
    def __init__(self, path):
        self.path = path; self.n, self.maxf, self.rec = read_meta(path)
        self.buf = None; self._foff = 6
    def _ensure(self):
        if self.buf is None:
            self.buf = np.memmap(self.path, dtype=np.uint8, mode='r', shape=(self.n, self.rec))
    def __len__(self): return self.n
    def __getitem__(self, i):
        self._ensure(); rec = self.buf[i]
        t = rec[0:4].view(np.float32)[0]; sc, oc = int(rec[4]), int(rec[5])
        f = rec[self._foff:].view(np.uint16)
        return f[:sc].astype(np.int64), f[self.maxf:self.maxf+oc].astype(np.int64), np.float32(t)
    def __getstate__(self):
        d = self.__dict__.copy(); d['buf'] = None; return d

def collate(b):
    si=[]; so=[]; oi=[]; oo=[]; tg=[]; sc=oc=0
    for s,o,t in b:
        so.append(sc); oo.append(oc); si.append(s); oi.append(o)
        sc+=len(s); oc+=len(o); tg.append(t)
    si = np.concatenate(si); oi = np.concatenate(oi)
    return (torch.from_numpy(si), torch.tensor(so), torch.from_numpy(oi),
            torch.tensor(oo), torch.tensor(tg, dtype=torch.float32))
print("model + dataset ready")

In [ ]:
# 7. Train on the GPU
import time
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ds = PackedDataset('/content/train.bin')
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate,
                num_workers=2, drop_last=True, pin_memory=(dev=='cuda'), persistent_workers=True)
model = NNUE().to(dev); opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.MSELoss()
print(f"{len(ds):,} positions on {dev}, {len(dl)} batches/epoch")
for ep in range(EPOCHS):
    model.train(); run=0.0; seen=0; t0=time.time()
    for b,(si,so,oi,oo,tg) in enumerate(dl,1):
        si=si.to(dev); so=so.to(dev); oi=oi.to(dev); oo=oo.to(dev); tg=tg.to(dev)
        loss = lossf(model.win_prob(model(si,so,oi,oo)), tg)
        opt.zero_grad(); loss.backward(); opt.step()
        run+=loss.item(); seen+=tg.numel()
        if b % 100 == 0 or b == len(dl):
            print(f"epoch {ep+1}/{EPOCHS} batch {b}/{len(dl)} loss {run/b:.5f} {seen/(time.time()-t0):,.0f} pos/s")
    print(f"  epoch {ep+1} done in {time.time()-t0:.0f}s")
torch.save(model.state_dict(), '/content/net.pt'); print("saved net.pt")

In [ ]:
# 8. Export -> net.nnue (NNU1 format the engine loads)
import struct, numpy as np
def f32(t): return np.ascontiguousarray(t.detach().cpu().numpy(), dtype='<f4')
m = model.eval()
W1=f32(m.transformer.weight); b1=f32(m.b1)
W2=f32(m.l1.weight); b2=f32(m.l1.bias); W3=f32(m.l2.weight); b3=f32(m.l2.bias)
W4=f32(m.l3.weight).reshape(-1); b4=f32(m.l3.bias)
assert W1.shape==(NUM_FEATURES,ACC) and W2.shape==(L1,2*ACC) and W3.shape==(L2,L1)
with open('/content/net.nnue','wb') as fh:
    fh.write(b'NNU1'); fh.write(struct.pack('<4i', NUM_FEATURES, ACC, L1, L2))
    for a in (W1,b1,W2,b2,W3,b3,W4,b4): fh.write(a.tobytes())
import os; print("wrote net.nnue", round(os.path.getsize('/content/net.nnue')/1e6), "MB")

In [ ]:
# 9. Sanity: start position should be a small eval (near 0). Then download net.nnue.
sp = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
pcs, stm = parse_fen(sp)
import torch
si = torch.tensor(features(pcs,stm)); oi = torch.tensor(features(pcs,stm^1))
with torch.no_grad():
    out = model.cpu()(si, torch.tensor([0]), oi, torch.tensor([0]))
print("start-position eval (cp):", round(float(out[0]),1), " win prob:", round(float(torch.sigmoid(out/EVAL_SCALE)[0]),3))
from google.colab import files
files.download('/content/net.nnue')

## After it finishes
`net.nnue` downloads to your computer. Put it in the repo at **`E:\\Chess\\dist\\net.nnue`**.
Then we wire it into the WASM bridge + search on the engine side.

Quality knobs (Config cell): raise `LIMIT` toward 0 (all 31.6M) and `EPOCHS` for a
stronger net; lower them for a quicker run.
